In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

WHICH_DATASET = 2

dataset = pd.read_csv(f'datasets/yjmob100k-dataset{WHICH_DATASET}.csv')
if WHICH_DATASET == 2:    
    dataset = dataset[dataset['d'] < 60]
else:
    dataset = dataset[dataset['d'] != 27]

In [ ]:
value_counts = None

def compute_distance_stats(df):
    global value_counts
    print("Sort values")
    df = df.sort_values(by=["uid", "d", "t"])

    print("Grouping by")
    df["prev_x"] = df.groupby("uid")["x"].shift()
    df["prev_y"] = df.groupby("uid")["y"].shift()
    df["prev_t"] = df.groupby("uid")["t"].shift()
    df["prev_d"] = df.groupby("uid")["d"].shift()

    print("Computing time step and distance")
    df["time_step"] = (df["d"] - df["prev_d"]) * 48 + (df["t"] - df["prev_t"])
    df["distance"] = np.sqrt((df["x"] - df["prev_x"])**2 + (df["y"] - df["prev_y"])**2)

    value_counts = df["time_step"].value_counts()
    print(value_counts.sort_index())

    # Drop rows with nulls (first row of each user/day)
    df = df.dropna(subset=["time_step", "distance"])

    print("Computing mean")
    # Group by timestep gap and compute average distance
    result = df.groupby("time_step")["distance"].mean().reset_index()
    result = result.sort_values("time_step")

    return result

distance_stats = compute_distance_stats(dataset)

In [ ]:
%matplotlib ipympl
import pandas as pd
import matplotlib.pyplot as plt

plt.figure()
plt.plot(distance_stats['time_step'], distance_stats['distance'] * 0.5)
plt.xlabel('Time Step')
plt.ylabel('Average Distance (km)')
plt.title('Average Distance per Time Step')
plt.grid(True)
plt.show()

plt.figure()
plt.plot(distance_stats['time_step'], value_counts)
plt.title("Gaps per timestep")
plt.xlabel("Time step")
plt.ylabel("Number of instances with this gap")
plt.yscale('log')


In [ ]:
output_file = f"datasets/yjmob100k-dataset{WHICH_DATASET}-interpolated.csv"
with open(output_file, "w") as f:
    f.write("uid,d,t,x,y,interpolated\n")

dataset["interpolated"] = False

for uid, group in tqdm(dataset.groupby("uid")):
    group = group.reset_index(drop=True)
    rows = []

    for i in range(1, len(group)):
        prev = group.loc[i - 1]
        curr = group.loc[i]

        # Compute full timestep difference
        delta = (curr["d"] - prev["d"]) * 48 + (curr["t"] - prev["t"])

        if 1 < delta < 48:
            # Add the previous row
            rows.append(prev.to_dict())

            # Interpolate missing steps
            for k in range(1, delta):
                frac = k / delta
                interp_x = prev["x"] + frac * (curr["x"] - prev["x"])
                interp_y = prev["y"] + frac * (curr["y"] - prev["y"])
                interp_t_total = (prev["d"] * 48 + prev["t"]) + k
                interp_d, interp_t = divmod(interp_t_total, 48)

                rows.append({
                    "uid": uid,
                    "d": int(interp_d),
                    "t": int(interp_t),
                    "x": interp_x,
                    "y": interp_y,
                    "interpolated": True
                })
        else:
            # No interpolation needed, just add the previous row
            rows.append(prev.to_dict())

    # Don't forget to add the last row
    rows.append(group.iloc[-1].to_dict())
    chunk_dataset = pd.DataFrame(rows)
    chunk_dataset.to_csv(output_file, mode="a", header=False, index=False)